In [ ]:
"""
Graficas SHAP en el espacio de features seleccionadas (post poly + SelectKBest).Letras y puntos mas grandes, incluye dependence plots, beeswarm agrupado y
completo, y heatmap.
Requiere: pip install shap
"""

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap
# config
RESULTS_ROOT = "resultados_8modelos"
METHOD       = "optuna"
MODEL_NAME   = "XGBoost"

TOP_N_DEPENDENCE = 10     # cuantas dependence plots generar (antes 5)
BEESWARM_MAX_DISPLAY = 12 # cuantas variables mostrar en el beeswarm "agrupado"

# tamanos de letra y elementos, mas grandes para que se lean sin zoom
FONT_TITLE  = 22
FONT_AXIS   = 18
FONT_TICK   = 16
FONT_LEGEND = 16
DOT_SIZE    = 90   # tamano de punto en dependence plots

plt.rcParams.update({
    "font.size":        FONT_AXIS,
    "axes.titlesize":   FONT_TITLE,
    "axes.labelsize":   FONT_AXIS,
    "xtick.labelsize":  FONT_TICK,
    "ytick.labelsize":  FONT_TICK,
    "legend.fontsize":  FONT_LEGEND,
    "figure.titlesize": FONT_TITLE,
})

SPLIT_PATH =os.path.join(RESULTS_ROOT, "particion_datos", "train_test_split.pkl")
MODEL_PATH =os.path.join(RESULTS_ROOT, METHOD, "models", f"{MODEL_NAME}.pkl")
SHAP_DIR =os.path.join(RESULTS_ROOT, METHOD, "shap")
os.makedirs(SHAP_DIR, exist_ok=True)

#cargar split y pipeline
split = joblib.load(SPLIT_PATH)
X_train, X_test = split["X_train"], split["X_test"]

pipeline = joblib.load(MODEL_PATH)

poly     = pipeline.named_steps["poly"]
scaler   = pipeline.named_steps["scaler"]
selector = pipeline.named_steps["select"]
model    = pipeline.named_steps["model"]

# reconstruir los nombres de las features seleccionadas
poly_names  = poly.get_feature_names_out(X_train.columns)
selected_mask  = selector.get_support()
selected_names  = poly_names[selected_mask]
n_selected      = selected_mask.sum()

print(f"k = {selector.k}  ->  {n_selected} features seleccionadas")
print(list(selected_names))

# transformar X_test hasta justo antes del modelo
def transform_to_model_space(X):
    Xt = poly.transform(X)
    Xt = scaler.transform(Xt)
    Xt = selector.transform(Xt)
    return pd.DataFrame(Xt, columns=selected_names, index=X.index)

X_test_sel = transform_to_model_space(X_test)

# shap con treeexplainer
explainer = shap.TreeExplainer(model)
shap_expl = explainer(X_test_sel)

# graficas
# beeswarm agrupado (top variables + "sum of N other features")
plt.figure(figsize=(11, 0.55 * BEESWARM_MAX_DISPLAY + 2))
shap.plots.beeswarm(shap_expl, max_display=BEESWARM_MAX_DISPLAY, show=False)
plt.title(f"SHAP Beeswarm — {MODEL_NAME} (top {BEESWARM_MAX_DISPLAY})", fontsize=FONT_TITLE, fontweight="bold")
plt.xlabel("Impacto en la prediccion (valor SHAP)", fontsize=FONT_AXIS)
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_beeswarm_top{BEESWARM_MAX_DISPLAY}.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print(f"Guardada: beeswarm_top{BEESWARM_MAX_DISPLAY}.png")

# beeswarm completo (todas las variables seleccionadas, sin agrupar)
plt.figure(figsize=(11, 0.55 * n_selected + 2))
shap.plots.beeswarm(shap_expl, max_display=n_selected, show=False)
plt.title(f"SHAP Beeswarm — {MODEL_NAME} (todas las {n_selected} variables)", fontsize=FONT_TITLE, fontweight="bold")
plt.xlabel("Impacto en la prediccion (valor SHAP)", fontsize=FONT_AXIS)
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_beeswarm_completo.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print("Guardada: beeswarm_completo.png")

# bar plot (importancia media |SHAP|), todas las variables
plt.figure(figsize=(11, 0.5 * n_selected + 2))
shap.plots.bar(shap_expl, max_display=n_selected, show=False)
plt.title(f"Importancia SHAP media — {MODEL_NAME}", fontsize=FONT_TITLE, fontweight="bold")
plt.xlabel("|Valor SHAP| promedio", fontsize=FONT_AXIS)
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_bar.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Guardada: shap_bar.png")

# heatmap plot (instancias x variables, ordenado por clustering)
plt.figure(figsize=(12, 0.35 * n_selected + 3))
shap.plots.heatmap(shap_expl, show=False)
plt.title(f"SHAP Heatmap — {MODEL_NAME}", fontsize=FONT_TITLE, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_shap_heatmap.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Guardada: shap_heatmap.png")

# dependence plots top-N, puntos mas grandes
mean_abs_shap = np.abs(shap_expl.values).mean(axis=0)
top_n = min(TOP_N_DEPENDENCE, n_selected)
top_idx = np.argsort(mean_abs_shap)[::-1][:top_n]

for rank, i in enumerate(top_idx, start=1):
    feat = selected_names[i]
    fig, ax = plt.subplots(figsize=(8, 6))
    shap.plots.scatter(shap_expl[:, feat], color=shap_expl, show=False, ax=ax)
    ax.set_title(f"SHAP Dependence — {feat}\n({MODEL_NAME}, rank #{rank})",
                 fontsize=FONT_TITLE - 2, fontweight="bold")
    ax.set_xlabel(feat, fontsize=FONT_AXIS)
    ax.set_ylabel("Valor SHAP", fontsize=FONT_AXIS)
    ax.tick_params(labelsize=FONT_TICK)
    # agrandar los puntos del scatter (el primer PathCollection es la nube de puntos)
    for coll in ax.collections:
        coll.set_sizes([DOT_SIZE])
    plt.tight_layout()
    safe_feat = feat.replace(" ", "_x_").replace("^", "_pow").replace("/", "_")
    fig.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_dependence_{rank:02d}_{safe_feat}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

print(f"Guardadas {top_n} dependence plots: {[selected_names[i] for i in top_idx]}")

# waterfall de un caso individual
plt.figure(figsize=(10, 0.45 * n_selected + 2))
shap.plots.waterfall(shap_expl[0], show=False)
plt.title(f"SHAP Waterfall — {MODEL_NAME} (ejemplo test #0)", fontsize=FONT_TITLE, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SHAP_DIR, f"{MODEL_NAME}_SELECTED_waterfall_ejemplo.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Guardada: waterfall_ejemplo.png")

print(f"\nTodas las graficas quedaron en: {SHAP_DIR}")